<a href="https://colab.research.google.com/github/RatanakamonS/Stock_Price/blob/main/Optimize_CVaR_and_Classical_Portfolio_S%26P500.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import numpy as np
import pandas as pd
import yfinance as yf

from scipy.optimize import minimize, linprog
from scipy.stats import norm

In [3]:
# =========================================================
# 0) Helper: robust price extractor (yfinance รองรับ MultiIndex)
# =========================================================
def get_price_series(yf_df: pd.DataFrame) -> pd.Series:
    if yf_df is None or len(yf_df) == 0:
        raise ValueError("yfinance returned empty data.")

    # MultiIndex columns (บางครั้ง yfinance คืนมาเป็น multiindex)
    if isinstance(yf_df.columns, pd.MultiIndex):
        # พยายามหา level 0 ก่อน
        lv0 = list(map(str, yf_df.columns.get_level_values(0)))
        if "Adj Close" in lv0:
            s = yf_df["Adj Close"]
            return s.iloc[:, 0] if isinstance(s, pd.DataFrame) else s
        if "Close" in lv0:
            s = yf_df["Close"]
            return s.iloc[:, 0] if isinstance(s, pd.DataFrame) else s

        # ถ้าไม่เจอ ให้หา level 1
        lv1 = list(map(str, yf_df.columns.get_level_values(1)))
        if "Adj Close" in lv1:
            s = yf_df.xs("Adj Close", axis=1, level=1)
            return s.iloc[:, 0]
        if "Close" in lv1:
            s = yf_df.xs("Close", axis=1, level=1)
            return s.iloc[:, 0]

        raise KeyError("Cannot find 'Adj Close' or 'Close' in yfinance MultiIndex columns.")

    # Single index columns
    if "Adj Close" in yf_df.columns:
        return yf_df["Adj Close"]
    if "Close" in yf_df.columns:
        return yf_df["Close"]

    raise KeyError("Cannot find 'Adj Close' or 'Close' in yfinance columns.")

In [4]:
# =========================================================
# 1) Load assets (cleaned adjusted close from GitHub)
# =========================================================
URL_ASSETS = (
    "https://raw.githubusercontent.com/RatanakamonS/Stock_Price/"
    "bf86f765ebc2be3aedb395cfb06bdc6fd37e72b7/"
    "3yrs_clean_sp500_adjusted_close_prices.csv"
)

print("Loading asset prices from GitHub...")
prices = pd.read_csv(URL_ASSETS, index_col=0, parse_dates=True, dayfirst=True)
prices = prices.apply(pd.to_numeric, errors="raise")

asset_ret = prices.pct_change().dropna()
tickers = asset_ret.columns.tolist()

start = asset_ret.index.min()
end   = asset_ret.index.max()

print(f"Assets loaded: N={asset_ret.shape[1]}, T(full)={asset_ret.shape[0]}, range={start.date()} to {end.date()}")


Loading asset prices from GitHub...
Assets loaded: N=495, T(full)=752, range=2022-11-01 to 2025-10-30


In [5]:
# =========================================================
# 2) Load market index (^GSPC) and align dates
# =========================================================
print("Loading market index (^GSPC) from yfinance...")
gspc_df = yf.download("^GSPC", start=start, end=end + pd.Timedelta(days=1), progress=False)
gspc_px = get_price_series(gspc_df).dropna()
gspc_ret = gspc_px.pct_change().dropna()

common_dates = asset_ret.index.intersection(gspc_ret.index)
asset_ret = asset_ret.loc[common_dates]
gspc_ret  = gspc_ret.loc[common_dates]

T, N = asset_ret.shape
mu_market = float(gspc_ret.mean())

print(f"Aligned data: N={N}, T={T}")
print(f"mu_market (^GSPC) = {mu_market:.10f}")


Loading market index (^GSPC) from yfinance...


/tmp/ipython-input-4016036512.py:5: FutureWarning: YF.download() has changed argument auto_adjust default to True
  gspc_df = yf.download("^GSPC", start=start, end=end + pd.Timedelta(days=1), progress=False)


Aligned data: N=495, T=751
mu_market (^GSPC) = 0.0008092511


In [6]:
# =========================================================
# 3) Parameters + moments
# =========================================================
beta = 0.95
tail = 1 - beta

mu = asset_ret.mean().values            # (N,)
Sigma = asset_ret.cov().values          # (N,N)
R = asset_ret.values                    # (T,N)

In [7]:
# =========================================================
# 4) MODEL 1: Mean–Variance (Allow Short) via SLSQP
#    min w' Σ w
#    s.t. sum(w)=1, mu'w >= mu_market
# =========================================================
def mv_variance(w):
    return float(w @ Sigma @ w)

cons_mv = [
    {"type": "eq",   "fun": lambda w: np.sum(w) - 1.0},
    {"type": "ineq", "fun": lambda w: (mu @ w) - mu_market},
]

w0 = np.ones(N) / N

res_mv = minimize(
    mv_variance, w0,
    constraints=cons_mv,
    method="SLSQP",
    options={"maxiter": 10000, "ftol": 1e-12}
)

print("\n" + "="*80)
print("Solver Report: Mean–Variance (Allow Short)")
print(f"success : {res_mv.success}")
print(f"message : {res_mv.message}")
print("="*80)

if not res_mv.success:
    raise RuntimeError("Mean–Variance optimization failed. Try relaxing constraints or check data.")

w_mv = res_mv.x
var_mv = mv_variance(w_mv)

# Normal-based VaR/CVaR for MV portfolio (LOSS = -Return)
Rp_mv = R @ w_mv
mu_p = float(Rp_mv.mean())
sd_p = float(Rp_mv.std(ddof=1))

z_beta = norm.ppf(beta)
VaR_mv_loss  = (-mu_p) + sd_p * z_beta
CVaR_mv_loss = (-mu_p) + sd_p * (norm.pdf(z_beta) / tail)


Solver Report: Mean–Variance (Allow Short)
success : True
message : Optimization terminated successfully


In [8]:
# =========================================================
# 5) CVaR LP Builder (Rockafellar–Uryasev)
#    Variables: x = [w(1..N), u(1..T), alpha]
#
#    minimize: alpha + (1/(T*tail)) * sum(u_t)
#    s.t.      u_t >= -r_t'w - alpha
#              u_t >= 0
#              sum(w) = 1
#              mu'w >= mu_market
#              (optional) w_i >= 0  (long-only)
# =========================================================
def build_cvar_lp(R, mu, mu_market, T, N, tail, long_only=False):
    # objective: alpha + (1/(T*tail))*sum(u)
    c = np.zeros(N + T + 1)
    c[N:N+T] = 1.0 / (T * tail)   # u part
    c[-1] = 1.0                   # alpha

    # A_ub x <= b_ub
    # (1) u_t >= -r_t'w - alpha  <=>  -r_t'w - u_t - alpha <= 0
    # (2) mu'w >= mu_market      <=>  -mu'w <= -mu_market
    A_ub = np.zeros((T + 1, N + T + 1))
    b_ub = np.zeros(T + 1)

    for t in range(T):
        A_ub[t, :N]      = -R[t, :]
        A_ub[t, N + t]   = -1.0     # -u_t
        A_ub[t, -1]      = -1.0     # -alpha
        b_ub[t] = 0.0

    # minimum return
    A_ub[T, :N] = -mu
    b_ub[T] = -mu_market

    # A_eq x = b_eq  (budget)
    A_eq = np.zeros((1, N + T + 1))
    A_eq[0, :N] = 1.0
    b_eq = np.array([1.0])

    # bounds
    # w: free (allow short) or >=0 (long-only)
    # u: >=0
    # alpha: free
    if long_only:
        w_bounds = [(0.0, None)] * N
    else:
        w_bounds = [(None, None)] * N

    u_bounds = [(0.0, None)] * T
    alpha_bounds = [(None, None)]

    bounds = w_bounds + u_bounds + alpha_bounds

    return c, A_ub, b_ub, A_eq, b_eq, bounds


def unpack_lp_solution(x, N, T):
    w = x[:N]
    u = x[N:N+T]
    alpha = x[-1]
    return w, u, alpha


def solver_report(name, res, w, mu, mu_market, long_only=False, tol=1e-6):
    print("\n" + "="*80)
    print(f"Solver Report: {name}")
    print(f"success : {res.success}")
    print(f"status  : {res.status}")
    print(f"message : {res.message}")

    if not res.success or w is None:
        print("="*80)
        return

    sum_w = float(np.sum(w))
    ret_w = float(mu @ w)
    min_w = float(np.min(w))

    budget_ok = abs(sum_w - 1.0) <= tol
    return_ok = (ret_w + tol) >= float(mu_market)
    long_ok   = (min_w + tol) >= 0.0 if long_only else True

    print("-"*80)
    print(f"sum(w)       = {sum_w:.10f} | budget_ok = {budget_ok}")
    print(f"min(w)       = {min_w:.10f} | long_only_ok = {long_ok}")
    print(f"mu @ w       = {ret_w:.10f}")
    print(f"mu_market    = {float(mu_market):.10f}")
    print(f"return_slack = {ret_w - float(mu_market):.10f} | return_ok = {return_ok}")
    print(f"FEASIBLE(manual) = {bool(budget_ok and return_ok and long_ok)}")
    print("="*80)

In [9]:
# =========================================================
# 6) MODEL 2: CVaR (Allow Short) + diagnostics
# =========================================================
c, A_ub, b_ub, A_eq, b_eq, bounds_allow = build_cvar_lp(
    R=R, mu=mu, mu_market=mu_market, T=T, N=N, tail=tail, long_only=False
)

res_cvar_allow = linprog(
    c, A_ub=A_ub, b_ub=b_ub,
    A_eq=A_eq, b_eq=b_eq,
    bounds=bounds_allow, method="highs"
)

w_cvar_allow = u_allow = alpha_allow = None
if res_cvar_allow.success:
    w_cvar_allow, u_allow, alpha_allow = unpack_lp_solution(res_cvar_allow.x, N, T)

solver_report("CVaR (Allow Short)", res_cvar_allow, w_cvar_allow, mu, mu_market, long_only=False)

VaR_allow_loss  = float(alpha_allow) if res_cvar_allow.success else np.nan
CVaR_allow_loss = float(res_cvar_allow.fun) if res_cvar_allow.success else np.nan


Solver Report: CVaR (Allow Short)
success : True
status  : 0
message : Optimization terminated successfully. (HiGHS Status 7: Optimal)
--------------------------------------------------------------------------------
sum(w)       = 1.0000000000 | budget_ok = True
min(w)       = -0.8816577814 | long_only_ok = True
mu @ w       = 0.0106429387
mu_market    = 0.0008092511
return_slack = 0.0098336877 | return_ok = True
FEASIBLE(manual) = True


In [10]:
# =========================================================
# 7) MODEL 3: CVaR (Long-only) + diagnostics
# =========================================================
c2, A_ub2, b_ub2, A_eq2, b_eq2, bounds_long = build_cvar_lp(
    R=R, mu=mu, mu_market=mu_market, T=T, N=N, tail=tail, long_only=True
)

res_cvar_long = linprog(
    c2, A_ub=A_ub2, b_ub=b_ub2,
    A_eq=A_eq2, b_eq=b_eq2,
    bounds=bounds_long, method="highs"
)

w_cvar_long = u_long = alpha_long = None
if res_cvar_long.success:
    w_cvar_long, u_long, alpha_long = unpack_lp_solution(res_cvar_long.x, N, T)

solver_report("CVaR (Long-only)", res_cvar_long, w_cvar_long, mu, mu_market, long_only=True)

VaR_long_loss  = float(alpha_long) if res_cvar_long.success else np.nan
CVaR_long_loss = float(res_cvar_long.fun) if res_cvar_long.success else np.nan


Solver Report: CVaR (Long-only)
success : True
status  : 0
message : Optimization terminated successfully. (HiGHS Status 7: Optimal)
--------------------------------------------------------------------------------
sum(w)       = 1.0000000000 | budget_ok = True
min(w)       = 0.0000000000 | long_only_ok = True
mu @ w       = 0.0009098569
mu_market    = 0.0008092511
return_slack = 0.0001006059 | return_ok = True
FEASIBLE(manual) = True


In [11]:
# =========================================================
# 8) Summary tables
# =========================================================
summary = pd.DataFrame({
    "Model": [
        "Mean–Variance (Allow Short)",
        "CVaR (Allow Short)",
        "CVaR (Long-only)"
    ],
    "Objective": [
        var_mv,
        CVaR_allow_loss,
        CVaR_long_loss
    ],
    "VaR_0.95_Loss": [
        VaR_mv_loss,
        VaR_allow_loss,
        VaR_long_loss
    ],
    "CVaR_0.95_Loss": [
        CVaR_mv_loss,
        CVaR_allow_loss,
        CVaR_long_loss
    ],
    "Solver_success": [
        bool(res_mv.success),
        bool(res_cvar_allow.success),
        bool(res_cvar_long.success)
    ],
    "Solver_message": [
        str(res_mv.message),
        str(res_cvar_allow.message),
        str(res_cvar_long.message)
    ]
})

weights = pd.DataFrame({
    "Ticker": tickers,
    "w_MV": w_mv,
    "w_CVaR_allow": (w_cvar_allow if res_cvar_allow.success else np.nan),
    "w_CVaR_long":  (w_cvar_long  if res_cvar_long.success  else np.nan),
})

print("\n" + "="*80)
print("SUMMARY")
print("="*80)
print(summary.to_string(index=False))

print("\n" + "="*80)
print("WEIGHTS (first 15 rows)")
print("="*80)
print(weights.to_string(index=False))

# optional save
summary.to_csv("summary_objectives.csv", index=False)
weights.to_csv("weights_all_models.csv", index=False)


SUMMARY
                      Model  Objective  VaR_0.95_Loss  CVaR_0.95_Loss  Solver_success                                                  Solver_message
Mean–Variance (Allow Short)   0.000007       0.003644        0.004776            True                            Optimization terminated successfully
         CVaR (Allow Short)   0.000018       0.000018        0.000018            True Optimization terminated successfully. (HiGHS Status 7: Optimal)
           CVaR (Long-only)   0.009836       0.007587        0.009836            True Optimization terminated successfully. (HiGHS Status 7: Optimal)

WEIGHTS (first 15 rows)
Ticker      w_MV  w_CVaR_allow  w_CVaR_long
   MMM -0.019902      0.090103     0.000000
   AOS -0.027781     -0.103966     0.000000
   ABT -0.024083      0.031422     0.000000
  ABBV -0.011091      0.044033     0.000000
   ACN -0.001507      0.021758     0.000000
  ADBE -0.014434     -0.078595     0.000000
   AMD -0.003509      0.071993     0.000000
   AES  0.0249